# 02 · Entrenamiento principal InterGATE V6

Ejecuta o reanuda las cinco semillas preespecificadas. Cada semilla guarda
inmediatamente su checkpoint, grafo predictivo, red biológica, auditoría
de adaptación y modelo de fase 2. Las métricas de desarrollo se muestran
al terminar cada semilla; no son resultados externos.

No cambie hiperparámetros después de iniciar esta celda. `force=False`
reutiliza una semilla solo si su bundle V6 está completo y coincide con el
protocolo.


In [ ]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "intergate").is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Ejecute este notebook dentro del repositorio InterGATE.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from intergate.config import CFG
from intergate.runtime_setup import (
    assert_v6_protocol, assert_transport_protocol, prepare_runtime,
)

assert_v6_protocol(CFG)
assert_transport_protocol(CFG)
ARTIFACTS = Path(CFG.ARTIFACTS_ROOT)
RUN_ROOT = ARTIFACTS / "INTERGATE_V6"
RESULTS = PROJECT_ROOT / "results_v6"
RESULTS.mkdir(parents=True, exist_ok=True)


In [ ]:
from intergate.transport import (
    TransportConfig, run_transport_seed,
)

runtime = prepare_runtime(CFG, include_external=False)
transport = TransportConfig.from_global(CFG)
SEEDS = tuple(map(int, CFG.STABILITY_SEEDS))
RESUME = True
FORCE = False
print("Semillas:", SEEDS)
print("Salida:", RUN_ROOT)


In [ ]:
rows = []
for position, seed in enumerate(SEEDS, 1):
    print(f"\n===== semilla {position}/{len(SEEDS)}: {seed} =====")
    row = run_transport_seed(
        seed,
        save_root=str(ARTIFACTS),
        transport=transport,
        resume=RESUME,
        force=FORCE,
        artifact_name="INTERGATE_V6",
    )
    rows.append(row)
    visible = pd.DataFrame([row])[[
        "seed", "macro_f1", "bal_acc",
        "val_macro_f1_phase1_transport_consensus",
        "transport_best_validation_score_phase1",
        "transport_pairwise_jaccard_mean",
        "transport_median_rank_spearman",
        "transport_all_parameter_deltas_zero",
    ]]
    display(visible)
    if bool(row["transport_all_parameter_deltas_zero"]):
        raise RuntimeError(
            f"Semilla {seed}: la adaptación estructural es un no-op. "
            "No continúe ni interprete Jaccard=1 como robustez."
        )

development_results = pd.DataFrame(rows).sort_values("seed")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
development_results.to_csv(
    RUN_ROOT / "development_results.csv", index=False
)
display(development_results)


In [ ]:
required = [
    "meta.json", "graph_bundle.pt", "biology_graph_bundle.pt",
    "model_pruned.pt", "transport_support_summary.json",
    "transport_adaptation_audit.csv",
]
missing = [
    str(RUN_ROOT / f"seed_{seed}" / name)
    for seed in SEEDS for name in required
    if not (RUN_ROOT / f"seed_{seed}" / name).is_file()
]
if missing:
    raise FileNotFoundError("Bundles incompletos:\n- " + "\n- ".join(missing))
if set(development_results.evaluation_split) != {"development_validation"}:
    raise RuntimeError("Una métrica no procede de desarrollo.")
if development_results.external_opened.astype(bool).any():
    raise RuntimeError("El externo fue abierto durante el entrenamiento.")
print("Entrenamiento principal V6 completo y externo cerrado.")
